# Read LFP and Movement for SleepScoring

In [34]:
MAX_HOUR_IN_SECONDS = 60*60*4

In [35]:
from Global_functions import read_file_by_time_steps
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

# Other parameters about what data to read
# chanList = [-1] # list of channels to extract, by index in saved file 
tstep = 60
t_vec = np.arange(0, MAX_HOUR_IN_SECONDS, tstep)

spikeglx_folder = '//132.66.34.210/Pixel1/1_auditory_neuropixels_BarakH/20241225_C15_T1_NP2_-10dB_g0'
binFullPath_LFP = Path(find_files(spikeglx_folder, r'.*\.lf\.bin$')[0])

chan_LFP, sr_LFP = read_file_by_time_steps(binFullPath_LFP, t_vec, tstep, [120, 130, 140])
chan_LFP = np.concatenate(chan_LFP, axis=1)

In [36]:
# Plot the first of the extracted channels
tLFP = np.linspace(t_vec[0], t_vec[-1] + tstep, chan_LFP.shape[1])

fig, ax = plt.subplots(3, 1)
ax[0].plot(tLFP[0:int(60*sr_LFP)], chan_LFP[0,0:int(60*sr_LFP)])
ax[1].plot(tLFP[0:int(60*sr_LFP)], chan_LFP[1,0:int(60*sr_LFP)])
ax[2].plot(tLFP[0:int(60*sr_LFP)], chan_LFP[2,0:int(60*sr_LFP)])
plt.show(block=False)

In [37]:
VID_locations_path = Path(find_files(spikeglx_folder, r'video_loc_sync.npz')[0])
VID_location_data = np.load(VID_locations_path)
VID_locations_time = VID_location_data['t_vec']
VID_locations_com = VID_location_data['location_com']

VID_locations_com = np.sqrt(np.abs(VID_locations_com[1:, 0] - VID_locations_com[:-1, 0]) ** 2 + np.abs(VID_locations_com[1:, 1] - VID_locations_com[:-1, 1]) ** 2)
VID_locations_com = VID_locations_com[0:len(VID_locations_time)]
VID_locations_time = VID_locations_time[0:len(VID_locations_com)]

%matplotlib inline
# %matplotlib qt
plt.figure()
plt.plot(VID_locations_time, VID_locations_com, color='blue')
plt.axhline(np.mean(VID_locations_com) + 2 * np.std(VID_locations_com), color='red', linestyle='dashed')
# plt.imshow(VID_location_data['first_frame'])

RATIO_cm_per_pixel = 0.104

# Sample LFPs at 512 Hz and Movement at 512 Hz

Remove DC from signals and normalize

In [38]:
EEG = (chan_LFP[0,0:int(MAX_HOUR_IN_SECONDS*sr_LFP)] - np.min(chan_LFP[0,0:int(MAX_HOUR_IN_SECONDS*sr_LFP)])) / (np.max(chan_LFP[0,0:int(MAX_HOUR_IN_SECONDS*sr_LFP)]) - np.min(chan_LFP[0,0:int(MAX_HOUR_IN_SECONDS*sr_LFP)]))
EEG = EEG - np.mean(EEG)
tEEG = tLFP[0:int(MAX_HOUR_IN_SECONDS*sr_LFP)]

EMG = (VID_locations_com[(VID_locations_time < MAX_HOUR_IN_SECONDS)] - np.min(VID_locations_com[(VID_locations_time < MAX_HOUR_IN_SECONDS)])) / (np.max(VID_locations_com[(VID_locations_time < MAX_HOUR_IN_SECONDS)]) - np.min(VID_locations_com[(VID_locations_time < MAX_HOUR_IN_SECONDS)]))
tEMG = VID_locations_time[(VID_locations_time < MAX_HOUR_IN_SECONDS)]

In [39]:
fig, ax = plt.subplots(2, 1)
ax[0].plot(tEEG, EEG, color='blue')
ax[1].plot(tEMG, EMG, color='blue')

Interpolate movement at time vec of LFP

In [40]:
EMG_rs = np.interp(tEEG, tEMG, EMG)

In [41]:
fig, ax = plt.subplots(2, 1)
ax[0].plot(tEEG, EEG, color='blue')
ax[1].plot(tEEG, EMG_rs, color='blue')

Downsample both vectors to 512Hz

In [42]:
t512 = np.linspace(tEEG[0], tEEG[-1], int((tEEG[-1] - tEEG[0]) * 512))
f512 = 1 / (t512[1] - t512[0])
EEG_512 = np.interp(t512, tEEG, EEG)
EMG_512 = np.interp(t512, tEEG, EMG_rs)

In [43]:
fig, ax = plt.subplots(2, 1)
ax[0].plot(t512, EEG_512, color='blue')
ax[1].plot(t512, EMG_512, color='blue')

Save EEG & EMG signals

In [44]:
from scipy.io import savemat

EEGdic = {"EEG": EEG_512}
EMGdic = {"EMG": EMG_512}

savemat(spikeglx_folder + "/EEG.mat", EEGdic)
savemat(spikeglx_folder + "/EMG.mat", EMGdic)